In [5]:
import pandas as pd

"""
Extract data from raw csv dataset
"""

PATH = "../data/raw/news/cnbc_articles_final.csv"

df = pd.read_csv(PATH)
df

,url,title,published_at,author,body,status,error,rss_link,discovery_title,discovery_published,discovery_query
0,https://www.cnbc.com/2021/09/08/investing-vete...,"Singapore banks are 'full of value,' says vete...",2021-09-09T02:00:02+0000,Yen Nee Lee,NaN,failed,body terlalu pendek (0 karakter),https://news.google.com/rss/articles/CBMipgFBV...,"Singapore banks are 'full of value,' says vete...","Wed, 08 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics
1,https://www.cnbc.com/2021/09/07/guinea-coup-ra...,Guinea coup rattles iron ore and bauxite marke...,2021-09-07T10:37:21+0000,Elliot Smith,A military junta claimed to have seized contro...,success,NaN,https://news.google.com/rss/articles/CBMiswFBV...,Guinea coup rattles iron ore and bauxite marke...,"Tue, 07 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics
2,https://www.cnbc.com/2021/09/08/carlyle-ceo-ke...,Carlyle CEO Kewsong Lee on what's driving the ...,2021-09-08T15:01:03+0000,Leslie Picker,In this article\r\n(Click here to subscribe to...,success,NaN,https://news.google.com/rss/articles/CBMizwFBV...,Carlyle CEO Kewsong Lee on what's driving the ...,"Wed, 08 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics
3,https://www.cnbc.com/2021/09/02/chinas-regulat...,China's health-care sector could be Beijing's ...,2021-09-02T02:39:44+0000,Weizhen Tan,China's health-care sector will probably be th...,success,NaN,https://news.google.com/rss/articles/CBMiqwFBV...,China's health-care sector could be Beijing's ...,"Wed, 01 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics
4,https://www.cnbc.com/2021/09/10/us-biden-china...,"Biden, Xi discuss avoiding confrontation in se...",2021-09-10T04:36:17+0000,Evelyn Cheng,BEIJING — Chinese President Xi Jinping and U.S...,success,NaN,https://news.google.com/rss/articles/CBMijgFBV...,"Biden, Xi discuss avoiding confrontation in se...","Fri, 10 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics
...,...,...,...,...,...,...,...,...,...,...,...
9495,https://www.cnbc.com/2025/08/06/honda-motor-re...,"Honda Motor quarterly profit halves, missing e...",2025-08-06T06:45:59+0000,Dylan Butts,Japanese auto giant Honda 's first-quarter ope...,success,NaN,https://news.google.com/rss/articles/CBMirgFBV...,"Honda Motor quarterly profit halves, missing e...","Wed, 06 Aug 2025 07:00:00 GMT",site:cnbc.com war
9496,https://www.cnbc.com/2025/08/06/cnbc-daily-ope...,CNBC Daily Open: Little sign of Trump shifting...,2025-08-06T06:30:01+0000,Yeo Boon Ping,In this article\r\nU.S. President Donald Trump...,success,NaN,https://news.google.com/rss/articles/CBMiqAFBV...,CNBC Daily Open: Little sign of Trump shifting...,"Wed, 06 Aug 2025 07:00:00 GMT",site:cnbc.com war
9497,https://www.cnbc.com/2025/08/05/5-things-to-kn...,5 things to know before the Tuesday open: Pala...,2025-08-05T11:40:24+0000,Alex Harring,In this article\r\nHere are five key things in...,success,NaN,https://news.google.com/rss/articles/CBMipgFBV...,5 things to know before the Tuesday open: Pala...,"Tue, 05 Aug 2025 07:00:00 GMT",site:cnbc.com war
9498,https://www.cnbc.com/2025/08/04/indias-it-layo...,Why India’s IT sector is shedding jobs,2025-08-04T06:44:54+0000,Anniek Bao,"India's IT sector is shedding jobs, and it's r...",success,NaN,https://news.google.com/rss/articles/CBMipwFBV...,Why India’s IT sector is shedding jobs - CNBC,"Mon, 04 Aug 2025 07:00:00 GMT",site:cnbc.com war


In [2]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from googlenewsdecoder import gnewsdecoder

"""
Scrap html from url
"""

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; research scraper/1.0)"
}

def resolve_article_url(url):
    try:
        result = gnewsdecoder(
            url,
            interval=1,
            timeout=15
        )
        
        return result["decoded_url"]
    except Exception:
        pass

    return url

def scrape_html(url):
    try:
        response = requests.get(url, headers=headers, timeout=20)
        response.raise_for_status()
        return response.text, response.status_code, None
    except requests.RequestException as exc:
        return None, None, str(exc)

results = [None] * len(df)

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {
        executor.submit(scrape_html, resolve_article_url(url)): index
        for index, url in df["url"].items()
    }

    for future in as_completed(futures):
        results[futures[future]] = future.result()

df[["html", "scrape_http_status", "scrape_error"]] = pd.DataFrame(
    results,
    index=df.index,
    columns=["html", "scrape_http_status", "scrape_error"]
)

df

ImportError: Modest backend is deprecated since selectolax 1.0. The engine has not been updated since 2021, while lexbor has been available since 2021 and is actively developed. Modest is outdated, not maintained, contains bugs and does not follow modern HTML5 standards. Please use lexbor backend instead: `from selectolax.lexbor import LexborHTMLParser`. The APIs are nearly identical, so in most projects swapping the import is all that's needed.

In [ ]:
import cleaning
from bs4 import BeautifulSoup

"""
Clean the dataset, fill the title, body, and published_at
"""

def extract_and_clean(html):
    if not isinstance(html, str) or not html.strip():
        return None, pd.NaT

    soup = BeautifulSoup(html, "html.parser")

    # extract publish date
    time_tag = soup.find("time", attrs={"datetime": True})
    published_at = (
        pd.to_datetime(time_tag["datetime"], errors="coerce")
        if time_tag
        else pd.NaT
    )
    
    # extract author
    author_tag = soup.find("a", attrs={"class": "Author-authorName"})
    author = author_tag.getText(" ", strip=True) if author_tag else None
    
    # extract title
    title_tag = soup.find("title")
    title = title_tag.get_text(" ", strip=True) if title_tag else None

    for tag in soup(["script", "style", "noscript", "nav", "footer", "header"]):
        tag.decompose()

    article = soup.select_one("[class*='ArticleBody'], article")
    content = article or soup
    paragraphs = content.find_all("p")
    text = " ".join(
        paragraph.get_text(" ", strip=True)
        for paragraph in paragraphs
        if paragraph.get_text(strip=True)
    )

    if not text:
        text = content.get_text(" ", strip=True)

    # extract and clean body
    cleaner = getattr(cleaning, "clean_text", None)
    body = cleaner(text) if callable(cleaner) else " ".join(text.split())

    return author, title, body, published_at


# Process the HTML and save the extracted values to df
extracted = df["html"].map(extract_and_clean)
extracted = pd.DataFrame(
    extracted.tolist(),
    index=df.index,
    columns=["extracted_author", "extracted_title", "extracted_body", "extracted_published_at"],
)

df["author"] = extracted["extracted_author"].combine_first(df['author'])
df["title"] = extracted["extracted_title"].combine_first(df["title"])
df["body"] = extracted["extracted_body"].combine_first(df["body"])
df["published_at"] = extracted["extracted_published_at"].combine_first(
    pd.to_datetime(df["published_at"], errors="coerce")
)

df

,url,title,published_at,author,body,status,error,discovery_title,discovery_published,discovery_query,html,scrape_http_status,scrape_error
0,https://news.google.com/rss/articles/CBMiqwFBV...,China's regulatory crackdown on healthcare sec...,2021-09-02 02:39:44+00:00,Weizhen Tan,China's health-care sector will probably be th...,success,NaN,China's health-care sector could be Beijing's ...,"Wed, 01 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None
1,https://news.google.com/rss/articles/CBMijgFBV...,"U.S. Biden, China's Xi discuss avoiding confro...",2021-09-10 04:36:17+00:00,Evelyn Cheng,BEIJING — Chinese President Xi Jinping and U.S...,success,NaN,"Biden, Xi discuss avoiding confrontation in se...","Fri, 10 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None
2,https://news.google.com/rss/articles/CBMinAFBV...,"EU von der Leyen speech: defense union, climat...",2021-09-15 09:30:36+00:00,Silvia Amaro,"LONDON — There is a new international order, w...",success,NaN,EU chief calls for more military independence ...,"Wed, 15 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None
3,https://news.google.com/rss/articles/CBMijAFBV...,Op-ed: Will China's President Xi’s big bet pay...,2021-09-19 12:00:04+00:00,Frederick Kempe,Chinese President Xi Jinping is making the mos...,success,NaN,Op-ed: Will China's President Xi’s big bet pay...,"Sun, 19 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None
4,https://news.google.com/rss/articles/CBMipAFBV...,"Germany's golden age is over, most Europeans t...",2021-09-16 06:40:40+00:00,Holly Ellyatt,Under the leadership of Chancellor Angela Merk...,success,NaN,"As Merkel prepares to leave office, many think...","Thu, 16 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3695,https://news.google.com/rss/articles/CBMicEFVX...,NaN,NaT,NaN,NaN,success,NaN,Drone stocks rally after Trump orders tariffs ...,"Fri, 14 Aug 2026 07:00:00 GMT",site:cnbc.com geopolitics,NaN,NaN,None
3696,https://news.google.com/rss/articles/CBMipgFBV...,NaN,NaT,NaN,NaN,success,NaN,HSBC's Jain: 'Chaos is the norm' as shipping d...,"Mon, 17 Aug 2026 07:00:00 GMT",site:cnbc.com geopolitics,NaN,NaN,None
3697,https://news.google.com/rss/articles/CBMiqgFBV...,NaN,NaT,NaN,NaN,success,NaN,Top Wall Street analysts like these 3 dividend...,"Sun, 16 Aug 2026 07:00:00 GMT",site:cnbc.com geopolitics,NaN,NaN,None
3698,https://news.google.com/rss/articles/CBMijwFBV...,NaN,NaT,NaN,NaN,success,NaN,Japanese automakers vulnerable to one-two punc...,"Mon, 17 Aug 2026 07:00:00 GMT",site:cnbc.com geopolitics,NaN,NaN,None


In [ ]:
"""
Execute data filtering
"""

from filtering.text_filtering import filter_news
filtered_df = filter_news(df)

filtered_df

,url,title,published_at,author,body,status,error,discovery_title,discovery_published,discovery_query,html,scrape_http_status,scrape_error,geopolitical_matches
0,https://news.google.com/rss/articles/CBMiqwFBV...,China's regulatory crackdown on healthcare sec...,2021-09-02 02:39:44+00:00,Weizhen Tan,China's health-care sector will probably be th...,success,NaN,China's health-care sector could be Beijing's ...,"Wed, 01 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None,"{'title': [], 'body': ['defense'], 'discovery_..."
1,https://news.google.com/rss/articles/CBMijgFBV...,"U.S. Biden, China's Xi discuss avoiding confro...",2021-09-10 04:36:17+00:00,Evelyn Cheng,BEIJING — Chinese President Xi Jinping and U.S...,success,NaN,"Biden, Xi discuss avoiding confrontation in se...","Fri, 10 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None,"{'title': [], 'body': ['conflict', 'summit'], ..."
2,https://news.google.com/rss/articles/CBMinAFBV...,"EU von der Leyen speech: defense union, climat...",2021-09-15 09:30:36+00:00,Silvia Amaro,"LONDON — There is a new international order, w...",success,NaN,EU chief calls for more military independence ...,"Wed, 15 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None,"{'title': ['defense'], 'body': ['troops', 'def..."
3,https://news.google.com/rss/articles/CBMijAFBV...,Op-ed: Will China's President Xi’s big bet pay...,2021-09-19 12:00:04+00:00,Frederick Kempe,Chinese President Xi Jinping is making the mos...,success,NaN,Op-ed: Will China's President Xi’s big bet pay...,"Sun, 19 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None,"{'title': [], 'body': ['defense', 'military'],..."
4,https://news.google.com/rss/articles/CBMipAFBV...,"Germany's golden age is over, most Europeans t...",2021-09-16 06:40:40+00:00,Holly Ellyatt,Under the leadership of Chancellor Angela Merk...,success,NaN,"As Merkel prepares to leave office, many think...","Thu, 16 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None,"{'title': [], 'body': ['election'], 'discovery..."
5,https://news.google.com/rss/articles/CBMiqAFBV...,"Quad: Narendra Modi meets Kamala Harris, ahead...",2021-09-24 04:01:01+00:00,Saheli Roy Choudhury,Indian Prime Minister Narendra Modi met with U...,success,NaN,India's Modi meets Kamala Harris ahead of bila...,"Fri, 24 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None,"{'title': [], 'body': ['military', 'summit', '..."
7,https://news.google.com/rss/articles/CBMixwFBV...,The Taliban takeover of Afghanistan could resh...,2021-10-05 07:27:25+00:00,Elliot Smith,The Taliban's takeover of Afghanistan and the ...,success,NaN,The Taliban takeover of Afghanistan could resh...,"Tue, 05 Oct 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None,"{'title': [], 'body': ['troops', 'war', 'milit..."
8,https://news.google.com/rss/articles/CBMiogFBV...,Is the Chinese stock market a smarter buy than...,2021-10-06 13:52:31+00:00,Eric Rosenbaum,Financial advisors have been telling investors...,success,NaN,As Wall Street sees a Chinese stock market 'on...,"Wed, 06 Oct 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None,"{'title': [], 'body': ['summit'], 'discovery_t..."
9,https://news.google.com/rss/articles/CBMijAFBV...,Russia's economy under President Putin in charts,2021-10-11 05:18:54+00:00,Holly Ellyatt,"Love him or loathe him, there's no doubt that ...",success,NaN,5 charts show Russia's economic highs and lows...,"Mon, 11 Oct 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None,"{'

In [ ]:
"""
Execute data preprocessing
"""

from preprocessing.text_preprocessing import preprocess_documents

preprocessed_df = preprocess_documents(filtered_df)

preprocessed_df

,url,title,published_at,author,body,status,error,discovery_title,discovery_published,discovery_query,html,scrape_http_status,scrape_error,geopolitical_matches,text_clean_light,text_clean_classical,tokens_classical,unigrams,bigrams,text_final
0,https://news.google.com/rss/articles/CBMiqwFBV...,China's regulatory crackdown on healthcare sec...,2021-09-02 02:39:44+00:00,Weizhen Tan,China's health-care sector will probably be th...,success,NaN,China's health-care sector could be Beijing's ...,"Wed, 01 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None,"{'title': [], 'body': ['defense'], 'discovery_...",China's health-care sector will probably be th...,china health care sector probably next fall sc...,"[china, health, care, sector, probably, next, ...","[china, health, care, sector, probably, next, ...","[china_health, health_care, care_sector, secto...",China's health-care sector will probably be th...
1,https://news.google.com/rss/articles/CBMijgFBV...,"U.S. Biden, China's Xi discuss avoiding confro...",2021-09-10 04:36:17+00:00,Evelyn Cheng,BEIJING — Chinese President Xi Jinping and U.S...,success,NaN,"Biden, Xi discuss avoiding confrontation in se...","Fri, 10 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None,"{'title': [], 'body': ['conflict', 'summit'], ...",BEIJING — Chinese President Xi Jinping and U.S...,beijing chinese president jinping president jo...,"[beijing, chinese, president, jinping, preside...","[beijing, chinese, president, jinping, preside...","[beijing_chinese, chinese_president, president...",BEIJING — Chinese President Xi Jinping and U.S...
2,https://news.google.com/rss/articles/CBMinAFBV...,"EU von der Leyen speech: defense union, climat...",2021-09-15 09:30:36+00:00,Silvia Amaro,"LONDON — There is a new international order, w...",success,NaN,EU chief calls for more military independence ...,"Wed, 15 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None,"{'title': ['defense'], 'body': ['troops', 'def...","LONDON — There is a new international order, w...",london new international order competition fie...,"[london, new, international, order, competitio...","[london, new, international, order, competitio...","[london_new, new_international, international_...","LONDON — There is a new international order, w..."
3,https://news.google.com/rss/articles/CBMijAFBV...,Op-ed: Will China's President Xi’s big bet pay...,2021-09-19 12:00:04+00:00,Frederick Kempe,Chinese President Xi Jinping is making the mos...,success,NaN,Op-ed: Will China's President Xi’s big bet pay...,"Sun, 19 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None,"{'title': [], 'body': ['defense', 'military'],...",Chinese President Xi Jinping is making the mos...,chinese president jinping make audacious geopo...,"[chinese, president, jinping, make, audacious,...","[chinese, president, jinping, make, audacious,...","[chinese_president, president_jinping, jinping...",Chinese President Xi Jinping is making the mos...
4,https://news.google.com/rss/articles/CBMipAFBV...,"Germany's golden age is over, most Europeans t...",2021-09-16 06:40:40+00:00,Holly Ellyatt,Under the leadership of Chancellor Angela Merk...,success,NaN,"As Merkel prepares to leave office, many think...","Thu, 16 Sep 2021 07:00:00 GMT",site:cnbc.com geopolitics,"<!DOCTYPE html><html lang=""en"" prefix=""og=http...",200.0,None,"{'title': [], 'body': ['election'], 'discovery...",Under the leadership of Chancellor Angela Merk...,leadership chancellor angela merkel germany po...,"[leadership, chancellor, angela, merkel, germa...","[leadership, chancellor, angela, merkel, germa...","[leadership_chancellor, chancellor_angela, ang...",Under the leadership of Chancellor Angela Merk...
5,https://news.google.com/rss/articles/CBMiqAFBV...,"Quad: Naren

In [ ]:
"""
Save the preprocessed dataset
"""

preprocessed_df.to_csv("../data/processed/news_preprocessed.csv", index=False)